# Step 2: 도구(Tool)를 활용한 자율적 장기 메모리

1단계(수동 주입)의 한계를 극복하기 위해, **LLM에게 메모리 스토어에 접근할 수 있는 도구(Tool)를 제공**합니다. 
이제 챗봇은 대화 중에 새로운 사실을 알게 되면 스스로 판단하여 저장(Write)하고, 과거 기억이 필요할 때 스스로 도구를 꺼내어 검색(Read)하는 능동적인 에이전트로 진화합니다.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### 1. 장기 메모리 저장소(Store) 준비

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

### 2. 컨텍스트 및 메모리 스키마 정의

In [ ]:
from dataclasses import dataclass
from typing import TypedDict

# 실행 컨텍스트 (사용자 식별)
@dataclass
class Context:
    user_id: str
    app_name: str

# LLM이 도구를 통해 추출/저장해야 할 정보의 구조 (Schema)
class UserInfo(TypedDict):
    personal_info: str
    preference: str

### 3. 메모리 제어 도구(Tools) 생성
개발자가 주입하지 않고, 이 도구들을 LLM에게 넘겨줍니다.

In [ ]:
from langchain_core.runnables import RunnableConfig
from langchain.tools import tool, ToolRuntime
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import uuid

@tool
def get_user_info(runtime: Annotated[ToolRuntime, InjectedToolArg]) -> str:
    """
    현재 사용자의 정보를 장기 메모리에서 조회합니다. (LLM이 스스로 판단하여 호출)
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name
    
    # 네임스페이스를 기반으로 검색
    memories = runtime.store.search((user_id, app))
    if not memories:
        return "기록된 정보 없음"

    results = []
    for item in memories:
        data = item.value
        if "personal_info" in data:
            results.append(f"- 개인정보: {data['personal_info']}")
        if "preference" in data:
            results.append(f"- 선호도: {data['preference']}")

    return "\n".join(results) if results else "데이터 형식 불일치로 읽을 수 없음"

In [ ]:
@tool
def save_user_info(user_info: UserInfo, runtime: Annotated[ToolRuntime, InjectedToolArg]):
    """
    사용자의 새로운 정보를 장기 메모리에 저장하거나 업데이트합니다. 
    대화 중 새로운 사실이 발견되면 LLM이 스스로 판단하여 이 도구를 호출해 기억합니다.
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name
    store = runtime.store

    # 고유 ID를 발급하여 Store에 저장 (Key-Value 형식)
    memory_key = str(uuid.uuid4())
    store.put((user_id, app), memory_key, user_info)

    return f"정보가 안전하게 저장되었습니다. (ID: {memory_key})"

### 4. 에이전트 생성 및 자율 학습 테스트

In [ ]:
from langchain.agents import create_agent

# LLM에게 잊지 말고 도구를 쓰라고 넌지시 알려줍니다.
system_message = "당신은 사용자의 정보를 기억하는 비서입니다. 사용자가 자신의 정보를 말하면 반드시 'save_user_info' 도구를 사용하여 저장하세요."

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[get_user_info, save_user_info], # 직접 만든 메모리 제어 도구를 제공
    store=store,
    context_schema=Context,
    system_prompt=system_message
)

In [ ]:
# 테스트 1: 새로운 정보를 말해보기. LLM이 스스로 `save_user_info` 도구를 호출하는지 확인하세요.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나는 아이스 아메리카노를 좋아해"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)
response

In [ ]:
# 테스트 2: 과거의 정보를 물어보기. LLM이 스스로 `get_user_info` 도구를 호출하는지 확인하세요.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고 있는 모든 것을 말해줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)
response